# 00 - Give our agent a work badge

**Goal:** create or find the agent's real Microsoft Entra identity. No model calls, telemetry license, or Azure deployment is needed here.

Think of the **agent code** as a worker, the **agent identity** as its work badge, and a **permission** as a key that opens one door. A badge is not the worker and does not open every door.

| Part | What it does | Where it lives |
|---|---|---|
| Blueprint | Parent identity definition; holds the credential | Entra |
| Blueprint principal | The blueprint's presence in this tenant | Entra |
| Child agent identity | One badge for our agent | Entra |
| Running agent | Uses a model and tools | Notebook kernel on your computer |
| Agent 365 | Management and governance around an agent | Microsoft 365; separate onboarding |

**Workshop order:** this notebook -> [01 Permissions](./01_Agent_Permissions.ipynb) -> [02 Agent and Azure traces](./02_Agent_and_Azure_Traces.ipynb) -> [03 User delegation](./03_User_Delegation.ipynb) -> [04 Agent 365](./04_Agent365_Observability.ipynb).

Each notebook starts from a fresh kernel and includes its own required setup. Nothing here automatically creates cloud resources or grants consent.

## 1. Prepare your computer - once

**You do this.** Open this project in VS Code. Select the project's Python 3.12+ environment as the notebook kernel. Install [requirements-showcase.txt](./requirements-showcase.txt) into that environment if needed, then restart the kernel.

In a VS Code PowerShell terminal, from the project root, if you are using the existing project environment:

```powershell
.\.venv\Scripts\python.exe -m pip install -r .\Agent_365\requirements-showcase.txt
```

Open the private `Agent_365/.env`. If it does not exist, copy `.env.example` to `.env`. **Do not overwrite an existing file.** Never show this file in the customer demo; it contains a secret.

**Expected:** the kernel uses the environment where the workshop packages are installed. If imports fail later, check the selected kernel before changing code.

## 2. Create or find the blueprint - once, in Entra

**An authorized administrator/developer does this.** Your tenant must support Entra Agent ID. Creating blueprints needs the appropriate Agent ID role; adding a development secret needs Agent ID Administrator. Azure subscription Contributor is not the same permission.

1. Open [Entra admin center](https://entra.microsoft.com) in the demo tenant.
2. Go to **Entra ID > Agents > Agent blueprints**.
3. Reuse the workshop blueprint if one exists. Otherwise choose **New agent blueprint (Preview)**, name it `Workshop blueprint`, choose an accountable sponsor/owner, and create it.
4. Open its details. Record its **Application/client ID** and **Object ID** privately.

**Why:** the blueprint is the parent of our agent's badge. The current portal wizard creates its blueprint principal too. When provisioning through Graph instead, create that principal explicitly; do not create duplicates.

**Stop if:** the Agents menu or creation action is missing. Ask your tenant administrator about access/availability; an ordinary app registration is not a replacement blueprint.

[Microsoft's blueprint creation steps](https://learn.microsoft.com/entra/agent-id/create-blueprint)

## 3. Create or find the child agent identity - once

**The authorized blueprint owner/administrator does this.**

1. Go to **Entra ID > Agents > Agent identities**.
2. Reuse your existing workshop child. If none exists, select **New agent identity (Preview)**, choose the blueprint from step 2, name the identity `Workshop identity`, assign its sponsor/owner, and create it.
3. Open its details and check the parent blueprint.
4. Save its **client/app ID** and **Object ID**. They are different identifiers, not passwords.

**Why:** Entra assigns the real identity; we do not invent a GUID in Python. The notebook will ask Entra for tokens for this existing child. Creating an Agent Framework `Agent(...)` later does not create another Entra identity.

**Expected:** you can point to the child and its parent in Entra. This is not yet Agent 365 registration.

[Microsoft's child identity creation steps](https://learn.microsoft.com/entra/agent-id/create-delete-agent-identities)

## 4. Give the blueprint a development credential - once

**The authorized administrator does this, privately, before presenting.** Reuse an approved unexpired credential if available. The credential belongs to the blueprint, NOT the child. Use federation/certificates for production.

If a new local-development secret is approved, use PowerShell 7 with the Microsoft.Graph.Applications module. Install that module first if it is missing. Replace both placeholders below. Use the blueprint's **application object ID**, not its principal object ID.

```powershell
Connect-MgGraph -TenantId '<TENANT-ID>' -Scopes 'AgentIdentityBlueprint.AddRemoveCreds.All'
$credential = Add-MgApplicationPassword -ApplicationId '<BLUEPRINT-APPLICATION-OBJECT-ID>' `
    -PasswordCredential @{displayName='Local workshop'; endDateTime=(Get-Date).ToUniversalTime().AddDays(7)}
$credential.SecretText | Set-Clipboard
```

Paste the clipboard value into `A365_OBSERVABILITY_BLUEPRINT_SECRET` in the private `.env`; do not display it. Then clear the clipboard (`Set-Clipboard -Value ''`) and remove the PowerShell variable (`Remove-Variable credential`). Your tenant may require a shorter lifetime.

**Why:** the blueprint proves it is allowed to request a token for its child. An ID alone cannot authenticate. [Credential setup reference](https://learn.microsoft.com/entra/agent-id/create-blueprint#other-app-credentials)

## 5. Save the four identity settings - once

**You do this in the private `.env`, not a notebook cell.** The names contain `OBSERVABILITY` for compatibility, but these same identities also authenticate to MCP.

```dotenv
A365_TENANT_ID=<your-directory-tenant-ID>
A365_OBSERVABILITY_BLUEPRINT_ID=<blueprint-client-app-ID>
A365_OBSERVABILITY_AGENT_ID=<child-client-app-ID>
A365_OBSERVABILITY_BLUEPRINT_SECRET=<approved-blueprint-secret-value>
```

Keep the **child Object ID** separately for the administrator's permission grants in notebook 01. Do not put that object ID in `A365_OBSERVABILITY_AGENT_ID`.

**Important:** Microsoft's Observability API ID `9b975845-388f-4429-889e-eab1ef63949c` is a different ID: it identifies the destination API, not your child. You do not create or expose that Microsoft API.

## 6. Show the customer the identity map - each session

**Setup needed:** complete steps 1-5; start the kernel from the project root or `Agent_365`. This cell loads private settings but prints only public identifiers. Do not show the `.env` editor.

Compare the three printed IDs to the Entra detail pages. This is a configuration check, **not proof of authentication**. The real token and protected API checks come next.

In [ ]:
from pathlib import Path
import sys
from uuid import UUID
from dotenv import load_dotenv

root = Path.cwd().parent if Path.cwd().name == 'Agent_365' else Path.cwd()
sys.path.insert(0, str(root))
load_dotenv(root / 'Agent_365' / '.env', override=False)
from Agent_365.workshop_helpers import required_setting

for label, key in [('Tenant', 'A365_TENANT_ID'),
                   ('Blueprint client ID', 'A365_OBSERVABILITY_BLUEPRINT_ID'),
                   ('Child client ID', 'A365_OBSERVABILITY_AGENT_ID')]:
    print(f'{label}: {UUID(required_setting(key))}')

**Say:** "We have created the badge, not deployed the worker. Next we give this badge permission to one protected tool."

Continue to [01 - Agent permissions](./01_Agent_Permissions.ipynb). It does not need a model or Agent 365 telemetry licensing.